# Prueba de diversos metodos de clasificación

In [14]:
import pandas as pd
import numpy as np



## **Conjunto de Datos**

<p>
El conjunto de datos <code>Carseats</code>, original del paquete de R <code>ISLR</code> y accesible en Python a través de <code>statsmodels.datasets.get_rdataset</code>, contiene información sobre la venta de sillas infantiles en 400 tiendas distintas.
<br>
<br>
Para cada una de las 400 tiendas se han registrado 11 variables. La empresa busca que cada tienda supere una meta de facturación de 900 mil unidades monetarias por local. Definimos la variable respuesta $ Y $ como 1 si la facturación —precio por cantidad vendida— supera esa meta, y 0 en caso contrario. El objetivo es predecir qué locales la superarán.

In [29]:
import statsmodels.api as sm
carseats = sm.datasets.get_rdataset("Carseats", "ISLR")
datos = carseats.data
datos['meta_facturado'] = ((datos.Sales * datos.Price) > 900).astype(int)
datos = datos.drop(columns = 'Sales')

In [31]:
datos.head()

,CompPrice,Income,Advertising,Population,Price,ShelveLoc,Age,Education,Urban,US,meta_facturado
0,138,73,11,276,120,Bad,42,17,Yes,Yes,1
1,111,48,16,260,83,Good,65,10,Yes,Yes,1
2,113,35,10,269,80,Medium,59,12,Yes,Yes,0
3,117,100,4,466,97,Medium,55,14,Yes,Yes,0
4,141,64,3,340,128,Bad,38,13,Yes,No,0


In [32]:
print(carseats.__doc__)

======== ===============
Carseats R Documentation
======== ===============

Sales of Child Car Seats
------------------------

Description
~~~~~~~~~~~

A simulated data set containing sales of child car seats at 400
different stores.

Usage
~~~~~

.. code:: R

   Carseats

Format
~~~~~~

A data frame with 400 observations on the following 11 variables.

``Sales``
   Unit sales (in thousands) at each location

``CompPrice``
   Price charged by competitor at each location

``Income``
   Community income level (in thousands of dollars)

``Advertising``
   Local advertising budget for company at each location (in thousands
   of dollars)

``Population``
   Population size in region (in thousands)

``Price``
   Price company charges for car seats at each site

``ShelveLoc``
   A factor with levels ``Bad``, ``Good`` and ``Medium`` indicating the
   quality of the shelving location for the car seats at each site

``Age``
   Average age of the local population

``Education``
   Education level

# Preprocesado de datos

Primero dividiremos los datos en entrenamiento (*train*) y prueba (*test*). Usaremos el conjunto de entrenamiento para comparar modelos y seleccionar sus hiperparámetros mediante validación cruzada, reservando el conjunto de prueba para la evaluación final.

El preprocesamiento incluirá `OneHotEncoder` para las variables categóricas y `StandardScaler` para las numéricas. Aunque los árboles y Random Forest no requieren estandarización, la utilizaremos para mantener un único pipeline de preprocesamiento. Este se ajustará dentro de cada partición de entrenamiento de la validación cruzada para evitar fugas de información.

In [33]:
X = datos.drop(columns=['meta_facturado'])
y = datos['meta_facturado']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    random_state=545)

In [34]:
from sklearn.compose import make_column_selector as selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

categorical_columns = selector(dtype_include=object)(X_train)
numerical_columns = selector(dtype_include=[int, float])(X_train)

categorical_columns, numerical_columns

(['ShelveLoc', 'Urban', 'US'],
 ['CompPrice',
  'Income',
  'Advertising',
  'Population',
  'Price',
  'Age',
  'Education'])

In [35]:
preprocessor = ColumnTransformer([
    ('onehot', OneHotEncoder(handle_unknown='ignore'), categorical_columns),
    ('estandar', StandardScaler(), numerical_columns)
])



In [36]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

In [37]:
grid = {
    'logistic': {
        'model': LogisticRegression(),
        'params': {
            'C': [0.1, 1, 10]
        }
    },
    'decision_tree': {
        'model': DecisionTreeClassifier(),
        'params': {
            'max_depth': [3, 5, 7]
        }
    },
    'random_forest': {
        'model': RandomForestClassifier(),
        'params': {
            'n_estimators': [10, 50, 100],
            'max_depth': [3, 5, 7]
        }
    }
}
X_transform = preprocessor.fit_transform(X_train)

grid_search_results = []
for model_name, model_info in grid.items():
    model = model_info['model']
    grid_search = GridSearchCV(model, model_info['params'], cv=5)
    grid_search.fit(X_transform, y_train)
    best_model = grid_search.best_estimator_
    best_score = grid_search.best_score_
    best_params = grid_search.best_params_
    
    grid_search_results.append({
        'model': model_name,
        'best_model': best_model,
        'best_score': best_score,
        'best_params': best_params
    })

In [38]:
grid_search_results = pd.DataFrame(grid_search_results)
grid_search_results

,model,best_model,best_score,best_params
0,logistic,LogisticRegression(C=0.1),0.850000,{'C': 0.1}
1,decision_tree,DecisionTreeClassifier(max_depth=3),0.730000,{'max_depth': 3}
2,random_forest,"(DecisionTreeClassifier(max_depth=5, max_featu...",0.796667,"{'max_depth': 5, 'n_estimators': 100}"


In [39]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

X_test_transform = preprocessor.transform(X_test)

resultados = []
for modelo in grid_search_results['best_model']:
    rdo = {}
    rdo['accuracy'] = accuracy_score(y_test, modelo.predict(X_test_transform))
    rdo['precision'] = precision_score(y_test, modelo.predict(X_test_transform))
    rdo['recall'] = recall_score(y_test, modelo.predict(X_test_transform))
    rdo['f1'] = f1_score(y_test, modelo.predict(X_test_transform))
    resultados.append(rdo)
pd.DataFrame(resultados, index=grid_search_results['model'])

,accuracy,precision,recall,f1
model,,,,
logistic,0.81,0.766667,0.657143,0.707692
decision_tree,0.75,0.656250,0.600000,0.626866
random_forest,0.80,0.777778,0.600000,0.677419


In [ ]:
log_reg = grid_search_results['best_model'].iloc[0]

coef = log_reg.coef_.ravel()
abs_coef = np.abs(coef)

pd.DataFrame(
    np.column_stack([coef, abs_coef]),
    index=preprocessor.get_feature_names_out(),
    columns=['coef', 'abs_coef']
)

ValueError: Shape of passed values is (1, 28), indices imply (1, 14)

In [41]:
importancia_predictores = pd.DataFrame({
    'Predictor': preprocessor.get_feature_names_out(),
    'Importancia': grid_search_results['best_model'][2].feature_importances_.round(3)
})

importancia_predictores.sort_values(by='Importancia', ascending=False)

,Predictor,Importancia
1,onehot__ShelveLoc_Good,0.198
7,estandar__CompPrice,0.186
12,estandar__Age,0.119
0,onehot__ShelveLoc_Bad,0.094
9,estandar__Advertising,0.075
11,estandar__Price,0.072
8,estandar__Income,0.067
10,estandar__Population,0.049
2,onehot__ShelveLoc_Medium,0.047
13,estandar__Education,0.038
